In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# read the merged enrichment datasets (generated from 2.1MergeEnrichmentResults_Regulator.ipynb)

In [ ]:
high_samples = pd.read_csv("./EnrichmentResults/PlausibleGenes_fromEnrichmentAnalysis_High.csv")
high_samples

In [ ]:
low_samples = pd.read_csv("./EnrichmentResults/PlausibleGenes_fromEnrichmentAnalysis_Low.csv")
low_samples

In [ ]:
# read motifdiff .diff files (generated from 2.2MotifDiff.ipynb)

In [ ]:
high_diff = pd.read_csv("./MotifDiff Results/_nucleotide_type.mono_score_type.NONE_mode_type_high.max.diff", sep="\t")
high_diff

In [ ]:
low_diff = pd.read_csv("./MotifDiff Results/_nucleotide_type.mono_score_type.NONE_mode_type_low.max.diff", sep="\t")
low_diff

In [ ]:
# check column names and drop 'Unnamed: 0'

In [ ]:
high_diff.columns

In [ ]:
high_diff = high_diff.drop(columns = ['Unnamed: 0'])
high_diff

In [ ]:
low_diff = low_diff.drop(columns = ['Unnamed: 0'])
low_diff

In [ ]:
# concatenate the variant-expression-pathway dataframes with motifdiff results 
# (in this case, since the variant-expression-pathway data was the input for motifdiff analysis, there was no shuffling of the rows, therefore direct merging was possible)

In [ ]:
df_merged_high = pd.concat([high_samples, high_diff], axis=1)
df_merged_high

In [ ]:
df_merged_low = pd.concat([low_samples, low_diff], axis=1)
df_merged_low

In [ ]:
# df_merged_high.to_csv("./MotifDiff Results/MotifDiff_Annotation_High.csv")

In [ ]:
# df_merged_low.to_csv("./MotifDiff Results/MotifDiff_Annotation_Low.csv")

#### integrating MotifDiff annotations with the variant-expression-pathway data

In [ ]:
# load dfs to merge - gene symbol and enrichment term 
go_df = pd.read_csv('./EnrichmentResults/LowVariants_Prioritized_GO_Enrichment.csv')
kegg_df = pd.read_csv('./EnrichmentResults/LowVariants_Prioritized_KEGG.csv')
reactome_df = pd.read_csv('')
variants_df = pd.read_csv('./Data_Files_Raw/PrioritizedVariants_Regulatory_Low.csv')

In [ ]:
def build_pathway_mapping(dfs_with_sources):
    pathway_records = []

    for df, source_name in dfs_with_sources:
        # skip if the dataframe is None or empty
        if df is None or df.empty:
            continue
            
        # ensures required columns exist before processing
        required_cols = ['geneID', 'Description', 'p.adjust']
        if not all(col in df.columns for col in required_cols):
            print(f"Warning: Skipping source '{source_name}' due to missing columns.")
            continue

        for _, row in df.iterrows():
            # Safely split genes
            genes = str(row['geneID']).split('/')
            term = row['Description']
            p_val = row['p.adjust']
            
            for gene in genes:
                gene_clean = gene.strip()
                if gene_clean: # Skip empty strings
                    pathway_records.append({
                        'SYMBOL': gene_clean,
                        'Pathway': term,
                        'Source': source_name,
                        'p_adjust': p_val
                    })

    # if no records were successfully processed
    if not pathway_records:
        return pd.DataFrame(columns=['SYMBOL', 'Pathway', 'Source', 'p_adjust'])

    # convert to dataframe
    df_pathways = pd.DataFrame(pathway_records)

    # sort by significance and drop duplicate gene-pathway links (keeping the best p-value)
    df_pathways = df_pathways.sort_values('p_adjust').drop_duplicates(
        subset=['SYMBOL', 'Pathway'], keep='first'
    )

    return df_pathways

In [ ]:
# example -> group datasets and run for available enrichment results | if one or more enrichment results are unavailable, comment it out as seen here 
datasets_high = [
    (go_df, 'GO_CC'),
    (kegg_df, 'KEGG'),
    #(reactome_df, 'Reactome')
]

df_pathways_high = build_pathway_mapping(datasets_high)

In [ ]:
# merge motifdiff results with pathway details 
df_combined_high = df_merged_high.merge(df_pathways_high, on='SYMBOL')
df_combined_high = df_combined_high.drop(columns = ['Unnamed: 0', 'consensus_score', 'supporting_tools'])
df_combined_high

In [ ]:
df_combined_high.to_csv("MotifDiff Results/DE_Annotation_MotifDiff_Pathway_High.csv", index = False)

In [ ]:
# example -> group datasets and run for available enrichment results | if one or more enrichment results are unavailable, comment it out as seen here 
datasets_low = [
    (go_df, 'GO_CC'),
    (kegg_df, 'KEGG'),
    (reactome_df, 'Reactome')
]

df_pathways_low = build_pathway_mapping(datasets_low)

In [ ]:
# merge motifdiff results with pathway details 
df_combined_low = df_merged_low.merge(df_pathways_low, on='SYMBOL')
df_combined_low = df_combined_low.drop(columns = ['Unnamed: 0', 'consensus_score', 'supporting_tools'])
df_combined_low

In [ ]:
df_combined_low.to_csv("MotifDiff Results/DE_Annotation_MotifDiff_Pathway_Low.csv", index = False)